In [1]:
import pandas as pd

In [ ]:
import joblib
import pandas as pd

# 모델 로드
model = joblib.load(r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\ML_model\logistic3_v2.pkl")

# 테스트 데이터 로드
test1 = pd.read_csv(r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\1차모델_테스트데이터셋.csv")

print('전체 컬럼:', test1.columns.tolist())

# 모델 학습 시 사용한 컬럼 순서 - 반드시 정확히 지정!
feature_list = [여기에_사용한_피처컬럼리스트_정확히_입력]  # 예: ['age', 'income', ...]

X_test1 = test1[feature_list]
y_test1 = test1['is_phishing']

print('X_test1 shape:', X_test1.shape)
print('y_test1 shape:', y_test1.shape)

y_pred1 = model.predict(X_test1)
print('y_pred1 shape:', y_pred1.shape)

In [ ]:
print(X_test1.shape)
print(y_pred1.shape)
print(y_test1.shape)

In [ ]:
df = pd.read_csv(r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\LLM_phishing_data_sorted.csv')

In [ ]:
df.head()

In [ ]:
# 각 파일별 포함된 피싱 타입(중복 제거)
phishing_type_per_file = df.groupby('file_name')['phishing_type'].unique().reset_index()
print(phishing_type_per_file)

In [ ]:
# text 길이를 나타내는 새로운 열 생성
df['text_length'] = df['text'].apply(len)

# 파일 네임별 text 길이 통계(describe)
text_length_stats_per_file = df.groupby('file_name')['text_length'].describe().reset_index()
text_length_stats_per_file

In [ ]:
# text_length 컬럼이 이미 존재한다고 가정
# 파일별 통계 대신 phishing_type별로 집계
text_length_stats_per_type = df.groupby('phishing_type')['text_length'].describe().reset_index()
print(text_length_stats_per_type)

In [ ]:
df['phishing_type'] = df['phishing_type'].replace('세금환급형', '기관사칭형')

In [ ]:
# text_length 컬럼 생성(이미 있을 경우 생략)
df['text_length'] = df['text'].apply(len)

# 피싱 타입별 통계 집계
stats_per_type = df.groupby('phishing_type')['text_length'].agg(
    count='count',           # 문장 개수
    char_sum='sum',          # 총 글자 개수
    mean='mean',             # 평균 글자수
    std='std',               # 표준편차
    min='min',
    q25=lambda x: x.quantile(0.25),
    median='median',
    q75=lambda x: x.quantile(0.75),
    max='max'
).reset_index()


In [ ]:
# 전체 문장 수, 전체 글자 수
total_count = df.shape[0]
total_char_sum = df['text_length'].sum()

# 각 타입별 문장 비율/글자 비율 계산
stats_per_type['count_ratio'] = stats_per_type['count'] / total_count * 100
stats_per_type['char_ratio'] = stats_per_type['char_sum'] / total_char_sum * 100

# 보기 쉽게 소수점 한 자리로 표시(선택 사항)
stats_per_type['count_ratio'] = stats_per_type['count_ratio'].round(1)
stats_per_type['char_ratio'] = stats_per_type['char_ratio'].round(1)


In [ ]:
stats_per_type

In [ ]:
# 1. 비율(%) 형식 맞추기 및 컬럼명 변경
stats_per_type['문장비율'] = stats_per_type['count_ratio'].map(lambda x: f"{x:.1f}%")
stats_per_type['글자비율'] = stats_per_type['char_ratio'].map(lambda x: f"{x:.1f}%")

# 2. 25%, 50%, 75% 컬럼명으로 변경
stats_per_type = stats_per_type.rename(
    columns={
        'q25': '25%',
        'median': '50%',
        'q75': '75%',
        'count_ratio': '문장비율(%)',
        'char_ratio': '글자비율(%)'
    }
)

# 3. 영어 비율 원본 컬럼 제거(선택)
stats_per_type = stats_per_type.drop(columns=['문장비율(%)', '글자비율(%)'])

# 4. 최종 확인
stats_per_type


In [ ]:
stats_per_type

In [ ]:
import joblib
from kiwipiepy import Kiwi

# 1. 모델 경로
MODEL_PATH = r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\pipeline_Ngram_kfold_lr.pkl'

# 2. 모델 불러오기 (joblib 사용, 벡터라이저 불필요)
ensemble_model = joblib.load(MODEL_PATH)

# 3. Kiwi 토크나이저 준비
kiwi = Kiwi()

# 4. 텍스트 전처리 함수
def tokenize_new_text(text):
    tokens = kiwi.tokenize(text)
    keywords = []
    for token in tokens:
        if token.tag in ['NNG']:
            keywords.append(token.form)
        elif token.tag in ['VV', 'VA']:
            keywords.append(token.lemma)
    return ' '.join(keywords)

# 5. 확률 기반 등급 함수
def classify_probability(prob):
    if prob < 0.3:
        return "✅ 일반 대화"
    elif prob < 0.7:
        return "⚠️ 보류 (판단 어려움)"
    else:
        return "🚨 보이스피싱 의심"

# 6. 예측 파이프라인 함수 (파이프라인 모델 기준)
def predict_text(text, model):
    processed_text = tokenize_new_text(text)
    prob = model.predict_proba([processed_text])[0][1]
    prediction = classify_probability(prob)
    print(f"\n👄 입력 텍스트:\n{text}")
    print(f"📊 보이스피싱 확률: {prob:.2f}")
    print(f"🧐 판단 결과: {prediction}\n")

# 7. 예시 문장
sample_texts = [
    "일반대화: 네, 여보세요. 네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요? 안녕하세요. 이번에 마이너스 통장 개설이 가능한지 문의드릴려고요. 네, 가능하십니다. 혹시 직장에 다니고 계신가요? 네, 정규직이고 4대 보험 다 가입돼 있어요. 네, 급여 이체 내역이 있으시면 심사에서 위뢰하시고요. 군무지는 혹시 어디신가요? 서울시청 근처 중소기업이에요. 그럼 방문하실 지점은 서울 광정 지점이 제일 가깝겠네요. 네, 지점에서 바로 개설되나요? 보통 30분 안에 가능합니다. 신분증과 재직증명서 급여명세서 지참해주시면 돼요. 금리는 보통 어느 정도에요. 현재 기준으로 연 4.2%입니다. 사용한 금액에 대해서만 미자가 붙어요. 한도는요. 신용에 따라 다릅니다. 하지만 고객님 소득 수준이면 2000만원 정도 예상이 됩니다. 그럼 준비해서 내일 방문 드릴게요. 네, 오실 때 대기시간 없도록 예약도 도와드릴게요. 성함 말씀해주시겠어요. 정영재요. 네, 알겠습니다.",
    "피싱대화: 네, 여보세요. 네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니다. 지금 급히 연락드린 건 고객님의 명의로 개통된 휴대폰이 금융사기 사건에 연루되었기 때문입니다. 예, 제 제 명이요. 그 그런 적이 없는데요. 네, 그럴 줄 알았습니다. 보통은 피해자분이 모르신 모르는 상태로 진행되거든요. 혹시 최근에 주민등록증이나 계좌번호 정보 유출된 적 있으신가요? 글 글쎄요. 아니, 개인 정보는 조심했는데요. 현재 피해 접수된 사건에서 고객님 명의 계좌가 범죄 자금 유통 경로로 사용된 정황이 표착됐습니다. 무슨 말이에요, 이게 제 계좌로요. 네, 그래서 계좌를 당장 확인해야 하고, 자산 보호를 위해서 금융강도원과 연계된 안정 계좌로 이체 후 확인 절차를 거치셔야 합니다. 이 무슨 이체 말하는 거예요? 그게 꼭 필요해요. 네, 이건 임시 조치고 고객님의 결백을 증명하는 가장 빠른 장문입니다. 일단 지금 은행에 앱 켜주시고요. 정말 아무 잘못 없어요 게. 고객님, 저희가 도와드릴 겁니다. 수사 협조에 응해주시면 피해 복구도 가능하십니다. 알겠습니다. 지금 일단 앱 켰어요. 좋습니다. 자, 이체 메뉴 이제 들어가시고요. 제가 말씀드리는 계좌번호로 진행을 하시면 됩니다. 네, 계좌번호를 불러주실래요. 네, 국민은행 4372-98. 잠시만요. 전산 다시 확인하겠습니다. 아니, 근데 이거 스티 아니야 뭐야 이게 지금. 고객님, 지금 통화 녹취되고 있습니다. 허위 신고 시 처벌을 받으실 수 있습니다. 제 죄송합니다. 계속 진행할게요.",
    "일반대화: 안녕하세요. KB 국민카드 고객센터입니다. 무엇을 도와드릴까요? 아니, 이번 달 카드값이 너무 많이 나와서요. 그 일부만 혹시 먼저 납부 가능할까요? 네, 고객님. 일부 결제 금액 UR 약정 서비스, 즉 리볼빙 신청이 가능합니다. 리볼빙이요, 처음 들어보는데 뭐에요 이게? 고객님이 이번 달 결제 금액 주 일부만 결제하시면 나머지는 다음 달로 이월되는 제도입니다. 그럼 혹시 이자 같은 게 좀 있나요? 네, 이월된 금액에는 연 7.5%의 이자가 부과됩니다. 지금 제가 안 낸 납부해야 될 금액이 120만원인데, 그럼 혹시 60만원 정도만 먼저 낼 수 있을까요? 가능합니다. 다만 월 단위로 계속 누적되면 이자 부담이 커질 수 있어요. 아이 그거는 한 번만 이용하고 바로 갚을 수 있어가지고 괜찮긴 한데 신청하려면 어떻게 해야 돼요, 이거? 본인인증만 해주시면 전화상으로 신청 접수 가능합니다. 인증은 어떻게 해요? 생년월일과 카드번호 뒤 네 자리 확인 후 ARS 인증으로 진행됩니다. 네, 그러면 지금 바로 할게요. 그 생년월일은 92년 4월 3일이고, 카드 뒷자리는 8291 에요. 네, 감사합니다. 인증 음성 안내 연결 드릴게요.",
    "피싱대화: 여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어. 누구야 목소리가 이상한데. 나 현준이 폰이 고장 나서 목소리가 좀 다르게 들리나 봐. 아니, 무슨 눈인데 그려. 지금 친구가 보증금 때문에 돈 문제가 터졌는데 급하게 돈이 필요하대, 그래서 내가 좀 대신 내줘야 될 거 같애. 지, 친구, 우즘 그건 니가 해결할 문제가 아닌 거 같은데. 아니야 이게 저번에 나 때문에 일이 더 커졌어. 제발 좀 도와줘, 진짜 급해. 아니, 얼마나 필요한데. 300만원 정도만 먼저 부탁해 계좌번호 불러줄게. 아니야. 그쪽으로. 엄마 나 지금 회사 앞이야. 회의 중이라서 오래 통화도 못해 그냥 계좌로 좀 부탁해. 진짜 현진이 맞아 요즘 이상해서 그래. 엄마, 지금 나 뭐 믿겠어, 진짜야 일 끝나고 설명할게 제발. 어",
    "피싱대화: 네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 인해 반순 처리 중입니다. 네, 아니, 주소 제대로 썼는데요. 지금 시스템에 등록된 주소가 불일치로 표시되거든요. 그러면 자동으로 반송 절차가 진행되고 이렇게 진행될 수 있어요. 아니, 그러면 다시 받을 수는 있는 거예요? 네, 하지만 고객님이 고객센터 접속 후 주소를 다시 인증해 주셔야 합니다. 제가 문자로 링크 보내드리겠습니다. 무슨 링크요? 배송 정보 재입력 사이트입니다. 본인 확인 절차를 걸친 후에 카드 인증도 필요할 수 있습니다. 아니, 카드는 그건 좀 이상한데요. 네, 고객님 이해합니다. 요즘 보안 때문에 신원 확인이 강화되었습니다. 저희도 불편한데 꼭 필요한 절차입니다. 아니, 그렇다고 카드 정보까지 네. 고객님 물건이 고가 상품이시죠. 2차 인증이 없으면 분실 처리될 수 있습니다. 아니, 그러면 빨리 해야 되긴 하겠네요. 혹시 어디로 접속하면 되나요? 네, 문자로 링크 보내드리겠습니다. 네, 알겠습니다."
]

# 8. 예측 실행
for text in sample_texts:
    predict_text(text, model=ensemble_model)


In [ ]:
import re
import pickle
from kiwipiepy import Kiwi

# 0. (학습시 썼던) 텍스트 정제 함수, stopwords, kiwi 인스턴스 반드시 정의
def clean_text_simple(text):
    text = re.sub(r"[^\w\s<>]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

stop_words = ["하", "있", "되"]
kiwi = Kiwi()

# 1. (학습 때 사용했던 tokenizer 함수 정확히 복사)
def my_kiwi_tokenizer(text):
    tokens = kiwi.tokenize(text)
    return [
        token.form
        for token in tokens
        if token.tag in ["NNG", "NNP", "VV", "VA", "XR"]
        and token.form not in stop_words
    ]

# 2. 모델 및 벡터라이저 경로
MODEL_PATH = r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\final_logistic_regression_model_6000_JJ.pkl'
VECTORIZER_PATH = r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\tfidf_vectorizer_JJ.pkl'

# 3. pickle로 모델과 벡터라이저 불러오기 (함수/객체 정의 후!!)
with open(MODEL_PATH, "rb") as f:
    ensemble_model = pickle.load(f)
with open(VECTORIZER_PATH, "rb") as f:
    vectorizer = pickle.load(f)

# 4. 예측에 사용하는 텍스트 전처리 함수 (학습시 tokenizer와 동일 로직 사용!)
def tokenize_new_text(text):
    # 필요하면 clean_text_simple(text) 추가
    tokens = kiwi.tokenize(text)
    return ' '.join([
        token.form
        for token in tokens
        if token.tag in ["NNG", "NNP", "VV", "VA", "XR"]
        and token.form not in stop_words
    ])

# 5. 확률 기반 등급 함수
def classify_probability(prob):
    if prob < 0.3:
        return "✅ 일반 대화"
    elif prob < 0.7:
        return "⚠️ 보류 (판단 어려움)"
    else:
        return "🚨 보이스피싱 의심"

# 6. 예측 파이프라인 함수
def predict_text(text, model, vectorizer):
    processed_text = tokenize_new_text(text)
    X = vectorizer.transform([processed_text])
    prob = model.predict_proba(X)[0][1]
    prediction = classify_probability(prob)
    print(f"\n👄 입력 텍스트:\n{text}")
    print(f"📊 보이스피싱 확률: {prob:.2f}")
    print(f"🧐 판단 결과: {prediction}\n")

# 7. 예시 문장
sample_texts = [
    "일반대화: 네, 여보세요. 네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요? 안녕하세요. 이번에 마이너스 통장 개설이 가능한지 문의드릴려고요. 네, 가능하십니다. 혹시 직장에 다니고 계신가요? 네, 정규직이고 4대 보험 다 가입돼 있어요. 네, 급여 이체 내역이 있으시면 심사에서 위뢰하시고요. 군무지는 혹시 어디신가요? 서울시청 근처 중소기업이에요. 그럼 방문하실 지점은 서울 광정 지점이 제일 가깝겠네요. 네, 지점에서 바로 개설되나요? 보통 30분 안에 가능합니다. 신분증과 재직증명서 급여명세서 지참해주시면 돼요. 금리는 보통 어느 정도에요. 현재 기준으로 연 4.2%입니다. 사용한 금액에 대해서만 미자가 붙어요. 한도는요. 신용에 따라 다릅니다. 하지만 고객님 소득 수준이면 2000만원 정도 예상이 됩니다. 그럼 준비해서 내일 방문 드릴게요. 네, 오실 때 대기시간 없도록 예약도 도와드릴게요. 성함 말씀해주시겠어요. 정영재요. 네, 알겠습니다.",
    "피싱대화: 네, 여보세요. 네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니다. 지금 급히 연락드린 건 고객님의 명의로 개통된 휴대폰이 금융사기 사건에 연루되었기 때문입니다. 예, 제 제 명이요. 그 그런 적이 없는데요. 네, 그럴 줄 알았습니다. 보통은 피해자분이 모르신 모르는 상태로 진행되거든요. 혹시 최근에 주민등록증이나 계좌번호 정보 유출된 적 있으신가요? 글 글쎄요. 아니, 개인 정보는 조심했는데요. 현재 피해 접수된 사건에서 고객님 명의 계좌가 범죄 자금 유통 경로로 사용된 정황이 표착됐습니다. 무슨 말이에요, 이게 제 계좌로요. 네, 그래서 계좌를 당장 확인해야 하고, 자산 보호를 위해서 금융강도원과 연계된 안정 계좌로 이체 후 확인 절차를 거치셔야 합니다. 이 무슨 이체 말하는 거예요? 그게 꼭 필요해요. 네, 이건 임시 조치고 고객님의 결백을 증명하는 가장 빠른 장문입니다. 일단 지금 은행에 앱 켜주시고요. 정말 아무 잘못 없어요 게. 고객님, 저희가 도와드릴 겁니다. 수사 협조에 응해주시면 피해 복구도 가능하십니다. 알겠습니다. 지금 일단 앱 켰어요. 좋습니다. 자, 이체 메뉴 이제 들어가시고요. 제가 말씀드리는 계좌번호로 진행을 하시면 됩니다. 네, 계좌번호를 불러주실래요. 네, 국민은행 4372-98. 잠시만요. 전산 다시 확인하겠습니다. 아니, 근데 이거 스티 아니야 뭐야 이게 지금. 고객님, 지금 통화 녹취되고 있습니다. 허위 신고 시 처벌을 받으실 수 있습니다. 제 죄송합니다. 계속 진행할게요.",
    "일반대화: 안녕하세요. KB 국민카드 고객센터입니다. 무엇을 도와드릴까요? 아니, 이번 달 카드값이 너무 많이 나와서요. 그 일부만 혹시 먼저 납부 가능할까요? 네, 고객님. 일부 결제 금액 UR 약정 서비스, 즉 리볼빙 신청이 가능합니다. 리볼빙이요, 처음 들어보는데 뭐에요 이게? 고객님이 이번 달 결제 금액 주 일부만 결제하시면 나머지는 다음 달로 이월되는 제도입니다. 그럼 혹시 이자 같은 게 좀 있나요? 네, 이월된 금액에는 연 7.5%의 이자가 부과됩니다. 지금 제가 안 낸 납부해야 될 금액이 120만원인데, 그럼 혹시 60만원 정도만 먼저 낼 수 있을까요? 가능합니다. 다만 월 단위로 계속 누적되면 이자 부담이 커질 수 있어요. 아이 그거는 한 번만 이용하고 바로 갚을 수 있어가지고 괜찮긴 한데 신청하려면 어떻게 해야 돼요, 이거? 본인인증만 해주시면 전화상으로 신청 접수 가능합니다. 인증은 어떻게 해요? 생년월일과 카드번호 뒤 네 자리 확인 후 ARS 인증으로 진행됩니다. 네, 그러면 지금 바로 할게요. 그 생년월일은 92년 4월 3일이고, 카드 뒷자리는 8291 에요. 네, 감사합니다. 인증 음성 안내 연결 드릴게요.",
    "피싱대화: 여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어. 누구야 목소리가 이상한데. 나 현준이 폰이 고장 나서 목소리가 좀 다르게 들리나 봐. 아니, 무슨 눈인데 그려. 지금 친구가 보증금 때문에 돈 문제가 터졌는데 급하게 돈이 필요하대, 그래서 내가 좀 대신 내줘야 될 거 같애. 지, 친구, 우즘 그건 니가 해결할 문제가 아닌 거 같은데. 아니야 이게 저번에 나 때문에 일이 더 커졌어. 제발 좀 도와줘, 진짜 급해. 아니, 얼마나 필요한데. 300만원 정도만 먼저 부탁해 계좌번호 불러줄게. 아니야. 그쪽으로. 엄마 나 지금 회사 앞이야. 회의 중이라서 오래 통화도 못해 그냥 계좌로 좀 부탁해. 진짜 현진이 맞아 요즘 이상해서 그래. 엄마, 지금 나 뭐 믿겠어, 진짜야 일 끝나고 설명할게 제발. 어",
    "피싱대화: 네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 인해 반순 처리 중입니다. 네, 아니, 주소 제대로 썼는데요. 지금 시스템에 등록된 주소가 불일치로 표시되거든요. 그러면 자동으로 반송 절차가 진행되고 이렇게 진행될 수 있어요. 아니, 그러면 다시 받을 수는 있는 거예요? 네, 하지만 고객님이 고객센터 접속 후 주소를 다시 인증해 주셔야 합니다. 제가 문자로 링크 보내드리겠습니다. 무슨 링크요? 배송 정보 재입력 사이트입니다. 본인 확인 절차를 걸친 후에 카드 인증도 필요할 수 있습니다. 아니, 카드는 그건 좀 이상한데요. 네, 고객님 이해합니다. 요즘 보안 때문에 신원 확인이 강화되었습니다. 저희도 불편한데 꼭 필요한 절차입니다. 아니, 그렇다고 카드 정보까지 네. 고객님 물건이 고가 상품이시죠. 2차 인증이 없으면 분실 처리될 수 있습니다. 아니, 그러면 빨리 해야 되긴 하겠네요. 혹시 어디로 접속하면 되나요? 네, 문자로 링크 보내드리겠습니다. 네, 알겠습니다."
]

# 8. 예측 실행
for text in sample_texts:
    predict_text(text, ensemble_model, vectorizer)


In [ ]:
import joblib
from kiwipiepy import Kiwi

# 1. 모델 경로
MODEL_PATH = r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\pipeline_stacking.pkl'

# 2. 모델 불러오기 (joblib 사용, 벡터라이저 불필요)
ensemble_model = joblib.load(MODEL_PATH)

# 3. Kiwi 토크나이저 준비
kiwi = Kiwi()

# 4. 텍스트 전처리 함수
def tokenize_new_text(text):
    tokens = kiwi.tokenize(text)
    keywords = []
    for token in tokens:
        if token.tag in ['NNG']:
            keywords.append(token.form)
        elif token.tag in ['VV', 'VA']:
            keywords.append(token.lemma)
    return ' '.join(keywords)

# 5. 확률 기반 등급 함수
def classify_probability(prob):
    if prob < 0.3:
        return "✅ 일반 대화"
    elif prob < 0.7:
        return "⚠️ 보류 (판단 어려움)"
    else:
        return "🚨 보이스피싱 의심"

# 6. 예측 파이프라인 함수 (파이프라인 모델 기준)
def predict_text(text, model):
    processed_text = tokenize_new_text(text)
    prob = model.predict_proba([processed_text])[0][1]
    prediction = classify_probability(prob)
    print(f"\n👄 입력 텍스트:\n{text}")
    print(f"📊 보이스피싱 확률: {prob:.2f}")
    print(f"🧐 판단 결과: {prediction}\n")

# 7. 예시 문장
sample_texts = [
    "일반대화: 네, 여보세요. 네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요? 안녕하세요. 이번에 마이너스 통장 개설이 가능한지 문의드릴려고요. 네, 가능하십니다. 혹시 직장에 다니고 계신가요? 네, 정규직이고 4대 보험 다 가입돼 있어요. 네, 급여 이체 내역이 있으시면 심사에서 위뢰하시고요. 군무지는 혹시 어디신가요? 서울시청 근처 중소기업이에요. 그럼 방문하실 지점은 서울 광정 지점이 제일 가깝겠네요. 네, 지점에서 바로 개설되나요? 보통 30분 안에 가능합니다. 신분증과 재직증명서 급여명세서 지참해주시면 돼요. 금리는 보통 어느 정도에요. 현재 기준으로 연 4.2%입니다. 사용한 금액에 대해서만 미자가 붙어요. 한도는요. 신용에 따라 다릅니다. 하지만 고객님 소득 수준이면 2000만원 정도 예상이 됩니다. 그럼 준비해서 내일 방문 드릴게요. 네, 오실 때 대기시간 없도록 예약도 도와드릴게요. 성함 말씀해주시겠어요. 정영재요. 네, 알겠습니다.",
    "피싱대화: 네, 여보세요. 네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니다. 지금 급히 연락드린 건 고객님의 명의로 개통된 휴대폰이 금융사기 사건에 연루되었기 때문입니다. 예, 제 제 명이요. 그 그런 적이 없는데요. 네, 그럴 줄 알았습니다. 보통은 피해자분이 모르신 모르는 상태로 진행되거든요. 혹시 최근에 주민등록증이나 계좌번호 정보 유출된 적 있으신가요? 글 글쎄요. 아니, 개인 정보는 조심했는데요. 현재 피해 접수된 사건에서 고객님 명의 계좌가 범죄 자금 유통 경로로 사용된 정황이 표착됐습니다. 무슨 말이에요, 이게 제 계좌로요. 네, 그래서 계좌를 당장 확인해야 하고, 자산 보호를 위해서 금융강도원과 연계된 안정 계좌로 이체 후 확인 절차를 거치셔야 합니다. 이 무슨 이체 말하는 거예요? 그게 꼭 필요해요. 네, 이건 임시 조치고 고객님의 결백을 증명하는 가장 빠른 장문입니다. 일단 지금 은행에 앱 켜주시고요. 정말 아무 잘못 없어요 게. 고객님, 저희가 도와드릴 겁니다. 수사 협조에 응해주시면 피해 복구도 가능하십니다. 알겠습니다. 지금 일단 앱 켰어요. 좋습니다. 자, 이체 메뉴 이제 들어가시고요. 제가 말씀드리는 계좌번호로 진행을 하시면 됩니다. 네, 계좌번호를 불러주실래요. 네, 국민은행 4372-98. 잠시만요. 전산 다시 확인하겠습니다. 아니, 근데 이거 스티 아니야 뭐야 이게 지금. 고객님, 지금 통화 녹취되고 있습니다. 허위 신고 시 처벌을 받으실 수 있습니다. 제 죄송합니다. 계속 진행할게요.",
    "일반대화: 안녕하세요. KB 국민카드 고객센터입니다. 무엇을 도와드릴까요? 아니, 이번 달 카드값이 너무 많이 나와서요. 그 일부만 혹시 먼저 납부 가능할까요? 네, 고객님. 일부 결제 금액 UR 약정 서비스, 즉 리볼빙 신청이 가능합니다. 리볼빙이요, 처음 들어보는데 뭐에요 이게? 고객님이 이번 달 결제 금액 주 일부만 결제하시면 나머지는 다음 달로 이월되는 제도입니다. 그럼 혹시 이자 같은 게 좀 있나요? 네, 이월된 금액에는 연 7.5%의 이자가 부과됩니다. 지금 제가 안 낸 납부해야 될 금액이 120만원인데, 그럼 혹시 60만원 정도만 먼저 낼 수 있을까요? 가능합니다. 다만 월 단위로 계속 누적되면 이자 부담이 커질 수 있어요. 아이 그거는 한 번만 이용하고 바로 갚을 수 있어가지고 괜찮긴 한데 신청하려면 어떻게 해야 돼요, 이거? 본인인증만 해주시면 전화상으로 신청 접수 가능합니다. 인증은 어떻게 해요? 생년월일과 카드번호 뒤 네 자리 확인 후 ARS 인증으로 진행됩니다. 네, 그러면 지금 바로 할게요. 그 생년월일은 92년 4월 3일이고, 카드 뒷자리는 8291 에요. 네, 감사합니다. 인증 음성 안내 연결 드릴게요.",
    "피싱대화: 여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어. 누구야 목소리가 이상한데. 나 현준이 폰이 고장 나서 목소리가 좀 다르게 들리나 봐. 아니, 무슨 눈인데 그려. 지금 친구가 보증금 때문에 돈 문제가 터졌는데 급하게 돈이 필요하대, 그래서 내가 좀 대신 내줘야 될 거 같애. 지, 친구, 우즘 그건 니가 해결할 문제가 아닌 거 같은데. 아니야 이게 저번에 나 때문에 일이 더 커졌어. 제발 좀 도와줘, 진짜 급해. 아니, 얼마나 필요한데. 300만원 정도만 먼저 부탁해 계좌번호 불러줄게. 아니야. 그쪽으로. 엄마 나 지금 회사 앞이야. 회의 중이라서 오래 통화도 못해 그냥 계좌로 좀 부탁해. 진짜 현진이 맞아 요즘 이상해서 그래. 엄마, 지금 나 뭐 믿겠어, 진짜야 일 끝나고 설명할게 제발. 어",
    "피싱대화: 네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 인해 반순 처리 중입니다. 네, 아니, 주소 제대로 썼는데요. 지금 시스템에 등록된 주소가 불일치로 표시되거든요. 그러면 자동으로 반송 절차가 진행되고 이렇게 진행될 수 있어요. 아니, 그러면 다시 받을 수는 있는 거예요? 네, 하지만 고객님이 고객센터 접속 후 주소를 다시 인증해 주셔야 합니다. 제가 문자로 링크 보내드리겠습니다. 무슨 링크요? 배송 정보 재입력 사이트입니다. 본인 확인 절차를 걸친 후에 카드 인증도 필요할 수 있습니다. 아니, 카드는 그건 좀 이상한데요. 네, 고객님 이해합니다. 요즘 보안 때문에 신원 확인이 강화되었습니다. 저희도 불편한데 꼭 필요한 절차입니다. 아니, 그렇다고 카드 정보까지 네. 고객님 물건이 고가 상품이시죠. 2차 인증이 없으면 분실 처리될 수 있습니다. 아니, 그러면 빨리 해야 되긴 하겠네요. 혹시 어디로 접속하면 되나요? 네, 문자로 링크 보내드리겠습니다. 네, 알겠습니다."
]

# 8. 예측 실행
for text in sample_texts:
    predict_text(text, model=ensemble_model)


In [ ]:
import joblib
import re
from kiwipiepy import Kiwi
from typing import List

# 1. KiwiTokenizer 클래스
class KiwiTokenizer:
    def __init__(self):
        self.kiwi = None
        self.stop_words = ["하", "있", "되"]

    def __setstate__(self, state):
        self.__dict__.update(state)
        if 'kiwi' not in self.__dict__:
            self.kiwi = None

    def __call__(self, text: str) -> str:
        if self.kiwi is None:
            self.kiwi = Kiwi()
            self.kiwi.add_user_word("대포통장", "NNP")
            self.kiwi.add_user_word("계좌번호", "NNP")

        cleaned_text = re.sub(r"[^\w\s<>]", " ", text)
        cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
        tokens = self.kiwi.tokenize(cleaned_text)

        keywords = []
        for token in tokens:
            if token.tag in ['NNG', 'NNP']:
                if token.form not in self.stop_words:
                    keywords.append(token.form)
            elif token.tag in ['VV', 'VA']:
                if token.lemma not in self.stop_words:
                    keywords.append(token.lemma)

        return ' '.join(keywords)


# 2. 모델 경로 및 로드
MODEL_PATH = r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\stacking_model_pipeline_6000_JJ.pkl'
ensemble_model = joblib.load(MODEL_PATH)

# 3. 등급 분류 함수
def classify_probability(prob):
    if prob < 0.3:
        return "✅ 일반 대화"
    elif prob < 0.7:
        return "⚠️ 보류 (판단 어려움)"
    else:
        return "🚨 보이스피싱 의심"

# 4. 예측 함수
def predict_text(text, model, tokenizer):
    processed_text = tokenizer(text)  # str 출력됨
    prob = model.predict_proba([processed_text])[0][1]
    prediction = classify_probability(prob)
    print(f"\n👄 입력 텍스트:\n{text}")
    print(f"📊 보이스피싱 확률: {prob:.2f}")
    print(f"🧐 판단 결과: {prediction}\n")

# 5. 샘플 테스트
sample_texts = [
    "일반대화: 네, 여보세요. 네, 안녕하세요. 국민은행 상담센터입니다. 무엇을 도와드릴까요? 안녕하세요. 이번에 마이너스 통장 개설이 가능한지 문의드릴려고요. 네, 가능하십니다. 혹시 직장에 다니고 계신가요? 네, 정규직이고 4대 보험 다 가입돼 있어요. 네, 급여 이체 내역이 있으시면 심사에서 위뢰하시고요. 군무지는 혹시 어디신가요? 서울시청 근처 중소기업이에요. 그럼 방문하실 지점은 서울 광정 지점이 제일 가깝겠네요. 네, 지점에서 바로 개설되나요? 보통 30분 안에 가능합니다. 신분증과 재직증명서 급여명세서 지참해주시면 돼요. 금리는 보통 어느 정도에요. 현재 기준으로 연 4.2%입니다. 사용한 금액에 대해서만 미자가 붙어요. 한도는요. 신용에 따라 다릅니다. 하지만 고객님 소득 수준이면 2000만원 정도 예상이 됩니다. 그럼 준비해서 내일 방문 드릴게요. 네, 오실 때 대기시간 없도록 예약도 도와드릴게요. 성함 말씀해주시겠어요. 정영재요. 네, 알겠습니다.",
    "피싱대화: 네, 여보세요. 네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니다. 지금 급히 연락드린 건 고객님의 명의로 개통된 휴대폰이 금융사기 사건에 연루되었기 때문입니다. 예, 제 제 명이요. 그 그런 적이 없는데요. 네, 그럴 줄 알았습니다. 보통은 피해자분이 모르신 모르는 상태로 진행되거든요. 혹시 최근에 주민등록증이나 계좌번호 정보 유출된 적 있으신가요? 글 글쎄요. 아니, 개인 정보는 조심했는데요. 현재 피해 접수된 사건에서 고객님 명의 계좌가 범죄 자금 유통 경로로 사용된 정황이 표착됐습니다. 무슨 말이에요, 이게 제 계좌로요. 네, 그래서 계좌를 당장 확인해야 하고, 자산 보호를 위해서 금융강도원과 연계된 안정 계좌로 이체 후 확인 절차를 거치셔야 합니다. 이 무슨 이체 말하는 거예요? 그게 꼭 필요해요. 네, 이건 임시 조치고 고객님의 결백을 증명하는 가장 빠른 장문입니다. 일단 지금 은행에 앱 켜주시고요. 정말 아무 잘못 없어요 게. 고객님, 저희가 도와드릴 겁니다. 수사 협조에 응해주시면 피해 복구도 가능하십니다. 알겠습니다. 지금 일단 앱 켰어요. 좋습니다. 자, 이체 메뉴 이제 들어가시고요. 제가 말씀드리는 계좌번호로 진행을 하시면 됩니다. 네, 계좌번호를 불러주실래요. 네, 국민은행 4372-98. 잠시만요. 전산 다시 확인하겠습니다. 아니, 근데 이거 스티 아니야 뭐야 이게 지금. 고객님, 지금 통화 녹취되고 있습니다. 허위 신고 시 처벌을 받으실 수 있습니다. 제 죄송합니다. 계속 진행할게요.",
    "일반대화: 안녕하세요. KB 국민카드 고객센터입니다. 무엇을 도와드릴까요? 아니, 이번 달 카드값이 너무 많이 나와서요. 그 일부만 혹시 먼저 납부 가능할까요? 네, 고객님. 일부 결제 금액 UR 약정 서비스, 즉 리볼빙 신청이 가능합니다. 리볼빙이요, 처음 들어보는데 뭐에요 이게? 고객님이 이번 달 결제 금액 주 일부만 결제하시면 나머지는 다음 달로 이월되는 제도입니다. 그럼 혹시 이자 같은 게 좀 있나요? 네, 이월된 금액에는 연 7.5%의 이자가 부과됩니다. 지금 제가 안 낸 납부해야 될 금액이 120만원인데, 그럼 혹시 60만원 정도만 먼저 낼 수 있을까요? 가능합니다. 다만 월 단위로 계속 누적되면 이자 부담이 커질 수 있어요. 아이 그거는 한 번만 이용하고 바로 갚을 수 있어가지고 괜찮긴 한데 신청하려면 어떻게 해야 돼요, 이거? 본인인증만 해주시면 전화상으로 신청 접수 가능합니다. 인증은 어떻게 해요? 생년월일과 카드번호 뒤 네 자리 확인 후 ARS 인증으로 진행됩니다. 네, 그러면 지금 바로 할게요. 그 생년월일은 92년 4월 3일이고, 카드 뒷자리는 8291 에요. 네, 감사합니다. 인증 음성 안내 연결 드릴게요.",
    "피싱대화: 여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어. 누구야 목소리가 이상한데. 나 현준이 폰이 고장 나서 목소리가 좀 다르게 들리나 봐. 아니, 무슨 눈인데 그려. 지금 친구가 보증금 때문에 돈 문제가 터졌는데 급하게 돈이 필요하대, 그래서 내가 좀 대신 내줘야 될 거 같애. 지, 친구, 우즘 그건 니가 해결할 문제가 아닌 거 같은데. 아니야 이게 저번에 나 때문에 일이 더 커졌어. 제발 좀 도와줘, 진짜 급해. 아니, 얼마나 필요한데. 300만원 정도만 먼저 부탁해 계좌번호 불러줄게. 아니야. 그쪽으로. 엄마 나 지금 회사 앞이야. 회의 중이라서 오래 통화도 못해 그냥 계좌로 좀 부탁해. 진짜 현진이 맞아 요즘 이상해서 그래. 엄마, 지금 나 뭐 믿겠어, 진짜야 일 끝나고 설명할게 제발. 어",
    "피싱대화: 네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 인해 반순 처리 중입니다. 네, 아니, 주소 제대로 썼는데요. 지금 시스템에 등록된 주소가 불일치로 표시되거든요. 그러면 자동으로 반송 절차가 진행되고 이렇게 진행될 수 있어요. 아니, 그러면 다시 받을 수는 있는 거예요? 네, 하지만 고객님이 고객센터 접속 후 주소를 다시 인증해 주셔야 합니다. 제가 문자로 링크 보내드리겠습니다. 무슨 링크요? 배송 정보 재입력 사이트입니다. 본인 확인 절차를 걸친 후에 카드 인증도 필요할 수 있습니다. 아니, 카드는 그건 좀 이상한데요. 네, 고객님 이해합니다. 요즘 보안 때문에 신원 확인이 강화되었습니다. 저희도 불편한데 꼭 필요한 절차입니다. 아니, 그렇다고 카드 정보까지 네. 고객님 물건이 고가 상품이시죠. 2차 인증이 없으면 분실 처리될 수 있습니다. 아니, 그러면 빨리 해야 되긴 하겠네요. 혹시 어디로 접속하면 되나요? 네, 문자로 링크 보내드리겠습니다. 네, 알겠습니다."
]

# 6. 실행
if __name__ == "__main__":
    tokenizer = KiwiTokenizer()
    for text in sample_texts:
        predict_text(text, model=ensemble_model, tokenizer=tokenizer)


In [1]:
!uv add yt-dlp

Resolved 285 packages in 3.66s
 Downloaded yt-dlp
Prepared 1 package in 4.77s
Installed 1 package in 242ms
 + yt-dlp==2025.7.21


In [4]:
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4" https://www.youtube.com/watch?v=Chxt_5-TFhY

[youtube] Extracting URL: https://www.youtube.com/watch?v=Chxt_5-TFhY
[youtube] Chxt_5-TFhY: Downloading webpage
[youtube] Chxt_5-TFhY: Downloading tv client config
[youtube] Chxt_5-TFhY: Downloading player 662eb823-main
[youtube] Chxt_5-TFhY: Downloading tv player API JSON
[youtube] Chxt_5-TFhY: Downloading ios player API JSON
[youtube] Chxt_5-TFhY: Downloading m3u8 information
[info] Chxt_5-TFhY: Downloading 1 format(s): 134+140
[download] Destination: 보이스피싱 실제 사례 모음 ⧸ YTN [Chxt_5-TFhY].f134.mp4

[download]   0.4% of  242.00KiB at  Unknown B/s ETA Unknown
[download]   1.2% of  242.00KiB at    2.00MiB/s ETA 00:00  
[download]   2.9% of  242.00KiB at    4.68MiB/s ETA 00:00
[download]   6.2% of  242.00KiB at    5.95MiB/s ETA 00:00
[download]  12.8% of  242.00KiB at    6.27MiB/s ETA 00:00
[download]  26.0% of  242.00KiB at    5.68MiB/s ETA 00:00
[download]  52.5% of  242.00KiB at    7.88MiB/s ETA 00:00
[download] 100.0% of  242.00KiB at    9.03MiB/s ETA 00:00
[download] 100% of  242.00Ki